# Partie 2 : étude principale pass@k, session 11 : H4, évaluation des modèles entraînés (ponts, T = 4)

Plan gelé : `docs/partie2/plan_analyse.md` (v1.0). Adaptateurs de la session 10 fusionnés dans Qwen2.5-3B-Instruct, puis protocole de la session 1 à l'identique (500 ponts, graine 42, n = 32, T = 4) : un modèle par GPU.

Notebook généré par `scripts/genere_notebooks_partie2.py` : le modifier là, pas ici.


In [ ]:
BRANCH = "partie2-passk"
N_QUESTIONS, N_SAMPLES, T, TEMP = 500, 32, 4, 0.7   # valeurs par défaut ; les options d'un run priment
MAX_TOKENS, MAX_MODEL_LEN = 1024, 8192   # figés au pilote v2 ; arrêt à </search> et </answer>
FILES = [
    [('/tmp/qwen2.5-3b-it-sft-ppo', [])],
    [('/tmp/qwen2.5-3b-it-sft-depart', [])],
]


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
from pathlib import Path
REPO = "vqlentinn/memoire-rl-search"
CLONE = Path("/tmp/memoire-rl-search")
if not CLONE.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, f"https://github.com/{REPO}.git", str(CLONE)], check=True)
print(subprocess.run(["git", "-C", str(CLONE), "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q vllm==0.33.0 pandas pyarrow huggingface_hub peft==0.21.2 2>&1 | tail -3
!pip freeze > /kaggle/working/pip_freeze.txt
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| gpus', torch.cuda.device_count())"
import sys
sys.path.insert(0, str(CLONE))
from data.hotpot_distractor import telecharger
PARQUET = telecharger("/tmp/hotpot")
print(PARQUET)

In [ ]:
import os, threading, time
WORK = Path("/kaggle/working")
(WORK / "logs").mkdir(exist_ok=True)
status, verrou = [], threading.Lock()

# Fusion des adaptateurs de la session 10 dans le modèle de départ (fp16), sur CPU, avant vLLM
import glob
for s in ("ppo", "depart"):
    cfg = sorted(glob.glob(f"/kaggle/input/**/lora_sft_{s}/adapter_config.json", recursive=True))
    assert cfg, f"adaptateur lora_sft_{s} introuvable"
    r = subprocess.run(["python", "-u", "scripts/train_sft.py", "--fusion", str(Path(cfg[0]).parent),
                        "--out", f"/tmp/qwen2.5-3b-it-sft-{s}"], cwd=CLONE, capture_output=True, text=True,
                       env=dict(os.environ, CUDA_VISIBLE_DEVICES=""))
    print(r.stdout[-1500:], r.stderr[-1500:])
    assert r.returncode == 0, f"fusion {s} en échec"


def localiser(o):
    """Un fichier attendu sous /kaggle/input peut être monté ailleurs (selon la version de Kaggle) : on le cherche."""
    if not o.startswith("/kaggle/input") or Path(o).exists():
        return o
    import glob
    f = sorted(glob.glob(f"/kaggle/input/**/{Path(o).name}", recursive=True))
    if not f:
        raise FileNotFoundError(o)
    print(f"{o} introuvable, utilisé : {f[0]}", flush=True)
    return f[0]

def file_de_runs(gpu, runs):
    env = dict(os.environ, VLLM_WORKER_MULTIPROC_METHOD="spawn", CUDA_VISIBLE_DEVICES=str(gpu))
    for model, opts in runs:
        opts = [localiser(o) for o in opts]
        nom = model.split("/")[-1] + "".join("_" + o.strip("-").replace("/", "of").replace(":", "-")
                                             for o in opts if not o.startswith("/kaggle"))
        cmd = ["python", "-u", "scripts/run_passk.py", "--model", model, "--type", "bridge",
               "--n-questions", str(N_QUESTIONS), "--n-samples", str(N_SAMPLES), "--max-turns", str(T),
               "--temperature", str(TEMP), "--data", PARQUET,
               "--max-tokens", str(MAX_TOKENS), "--max-model-len", str(MAX_MODEL_LEN),
               "--outdir", str(WORK / "results" / "partie2")] + opts
        t0 = time.time()
        with open(WORK / "logs" / f"{nom}.log", "a") as f:
            code = subprocess.run(cmd, cwd=CLONE, env=env, stdout=f, stderr=subprocess.STDOUT).returncode
        with verrou:
            status.append((nom, gpu, code, round(time.time() - t0)))
            print(f"GPU {gpu} : {nom} code {code}, {time.time() - t0:.0f} s", flush=True)

fils = [threading.Thread(target=file_de_runs, args=(g, runs)) for g, runs in enumerate(FILES)]
for f in fils: f.start()
for f in fils: f.join()


In [ ]:
import json
from collections import Counter
resume = {}
for fichier in sorted((WORK / "results" / "partie2").glob("*.jsonl")):
    L = [json.loads(l) for l in fichier.read_text().split("\n") if l.strip()]
    if not L:
        continue
    st = Counter(l["status"] for l in L)
    par_q = Counter(l["id"] for l in L)
    resume[fichier.stem] = {
        "trajectoires": len(L), "questions_completes": sum(v >= N_SAMPLES for v in par_q.values()),
        "em": round(sum(l["em"] for l in L) / len(L), 3), "statuts": dict(st),
        "tronquees": round(sum(l["truncated"] for l in L) / len(L), 3),
        "recherches": round(sum(l["n_search"] for l in L) / len(L), 2),
        "jetons": round(sum(l["gen_tokens"] for l in L) / len(L)),
    }
(WORK / "resume_s11.json").write_text(json.dumps({"status": status, "resume": resume}, indent=1, ensure_ascii=False))
print(json.dumps({"status": status, "resume": resume}, indent=1, ensure_ascii=False))
echecs = [s for s in status if s[2] != 0]
if echecs:
    raise RuntimeError(f"Runs en échec : {echecs}")
